Notes

- should I do a 5 fold average for test acc?

# Initial Data Loading

In [ ]:
import sys
print(sys.executable)


In [ ]:
!{sys.executable} -m pip install --upgrade --force-reinstall -vvv "numpy<2.0,>=1.26" pandas pyarrow


In [ ]:
pip install 

In [ ]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt

#open full set

df = pd.read_csv("2visit_1month_7_26_25_secondVisit_mech2.csv")
df

In [ ]:
# Get counts for each unique value in 'buckets'
print(df['buckets'].value_counts())


In [ ]:
for i in df.columns:
    print(i)

In [ ]:
#columns we dont want

cols_X = ["buckets","days_2_clearance",
          "days_2_firstvisit","high_total_sx_severity",
         "record_id","redcap_repeat_instance","injury_mechanism","high_total_sx_severity_diff"]

# Save 'days_2_clearance' separately before dropping it
days_2_clearance_series = df["days_2_clearance"]

# Machine Learning Models

## LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

import numpy as np

# Assume df and cols_X are already defined
X = df.drop(cols_X, axis=1)
y = df['buckets']

# 1. Split into train+val and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Define hyperparameter grid (distributions for sampling)
param_dist = {
    'num_leaves': [31, 50, 70],
    'max_depth': [-1, 10, 20, 30],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 500, 1000],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

n_iter = 50
param_list = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=42))

best_score = 0
best_params = None

# 4. Manual random search loop with progress tracking
for i, params in enumerate(param_list, start=1):
    model = lgb.LGBMClassifier(random_state=42, **params)
    model.fit(X_train, y_train)
    y_preds = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_preds)
    
    # Track progress
    print(f"[{i}/{n_iter}] Accuracy: {acc:.4f} | Best so far: {best_score:.4f}")
    
    if acc > best_score:
        best_score = acc
        best_params = params
        print(f"  🔹 New best found! {best_params}")

print("\n✅ Best hyperparameters found:", best_params)
print("✅ Best validation accuracy:", best_score)


In [ ]:
import lightgbm as lgb

# Recreate model with best parameters found earlier
best_model = lgb.LGBMClassifier(random_state=42, **best_params)

correct = 0
total = len(df)
X = df.drop(cols_X, axis=1)
y = df['buckets']

for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    X_test = X.iloc[[i]]       # keep as DataFrame
    y_test = y.iloc[[i]]       # keep as Series
    
    # Re-initialize model in each loop to avoid carrying state
    model = lgb.LGBMClassifier(random_state=42, **best_params)
    model.fit(X_train, y_train)
    
    pred = model.predict(X_test)[0]
    if pred == y_test.values[0]:
        correct += 1

accuracy = correct / total
accuracy


In [ ]:
accuracy

In [ ]:
results ##.77 - w/treatment present

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# Initialize accumulator
feature_importance_sum = np.zeros(X.shape[1])

for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    
    model = lgb.LGBMClassifier(random_state=42, **best_params)
    model.fit(X_train, y_train)
    
    # accumulate feature importance (gain)
    feature_importance_sum += model.booster_.feature_importance(importance_type="gain")

# Average importance across all LOO models
feature_importance_avg = feature_importance_sum / total

# Create a DataFrame
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance_avg
}).sort_values(by='importance', ascending=False)

# Top 20 features
top_20_features = feature_importance_df.head(20)


In [ ]:
top_20_features #without treatment present

In [ ]:
# - LGBM

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import confusion_matrix

# Recreate model with best parameters found earlier
best_model = lgb.LGBMClassifier(random_state=42, **best_params)

correct = 0
total = len(df)
X = df.drop(cols_X, axis=1)
y = df['buckets']

# initialize accumulator for feature importance
feature_importance_sum = np.zeros(X.shape[1])

# to store predictions and true labels for specificity
y_true_all = []
y_pred_all = []

for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    X_test = X.iloc[[i]]       # keep as DataFrame
    y_test = y.iloc[[i]]       # keep as Series
    
    # Re-initialize model in each loop to avoid carrying state
    model = lgb.LGBMClassifier(random_state=42, **best_params)
    model.fit(X_train, y_train)
    
    # accumulate feature importance from this model
    feature_importance_sum += model.booster_.feature_importance(importance_type="gain")
    
    # prediction for accuracy
    pred = model.predict(X_test)[0]
    y_true_all.append(y_test.values[0])
    y_pred_all.append(pred)

    if pred == y_test.values[0]:
        correct += 1

    # progress tracker (every 50 iterations and final step)
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"Progress: {i+1}/{total} ({(i+1)/total:.1%}) complete")


# === Specificity Calculation ===
# Assumes binary classification with labels 0 (negative) and 1 (positive)
tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()
specificity = tn / (tn + fp)

# print("\nLeave-One-Out Accuracy:", accuracy)

# print(feature_importance_df)

# Precision, Recall (Sensitivity), and F1 Score
# 'binary' average assumes y=1 is the positive class.
precision = precision_score(y_true_all, y_pred_all, average='binary', zero_division=0)
recall = recall = recall_score(y_true_all, y_pred_all, average='binary', zero_division=0) # Recall is the same as Sensitivity

# F1 Score
f1 = f1_score(y_true_all, y_pred_all, average='binary', zero_division=0)

# Calculate Specificity (True Negative Rate) and Sensitivity (True Positive Rate/Recall)
# Assumes binary classification with labels 0 (negative) and 1 (positive)
tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()


print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
#precision,recall, f1 (removal of treatment present) - 
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
# 'binary' average assumes y=1 is the positive class.
precision = precision_score(y_true_all, y_pred_all, average='binary', zero_division=0)
recall = recall = recall_score(y_true_all, y_pred_all, average='binary', zero_division=0) # Recall is the same as Sensitivity

# F1 Score
f1 = f1_score(y_true_all, y_pred_all, average='binary', zero_division=0)
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
# !pip install seaborn

In [ ]:
# confusion matrix table (removal of treatment present)
 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(y_true_all, y_pred_all)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion Matrix:\n{cm}")
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

# Plot confusion matrix
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0,1], yticklabels=[0,1])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - LightGBM')
plt.show()



In [ ]:
#specificity (removal of treatment present)
print("Specificity (True Negative Rate):", specificity)

In [ ]:
from sklearn.metrics import confusion_matrix

# Assumes binary classification with labels 0 (negative) and 1 (positive) (removal of treatment present)
tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()
sensitivity = tp / (tp + fn)  # True Positive Rate

print("Sensitivity (True Positive Rate):", sensitivity)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Get top feature names
top_features = top_20_features['feature'].tolist()

# Compute average effect per feature, handling empty groups
effects = []
for f in top_features:
    median = X[f].median()
    
    # Split data into high and low subsets
    X_high = X[X[f] > median]
    X_low  = X[X[f] <= median]
    
    # Compute predicted probabilities; replace empty group with 0
    prob_high = model.predict_proba(X_high)[:, 1].mean() if not X_high.empty else 0.0
    prob_low  = model.predict_proba(X_low)[:, 1].mean() if not X_low.empty else 0.0
    
    # Compute average effect
    avg_effect = prob_high - prob_low
    
    # Store result
    effects.append({
        'feature': f,
        'avg_effect': avg_effect,
        'direction': 'toward 1' if avg_effect > 0 else 'toward 0'
    })

# Create DataFrame from results
effects_df = pd.DataFrame(effects)

# Handle case where no valid effects were computed
if effects_df.empty:
    raise ValueError("No valid features found for effect calculation. Check your data and top_20_features list.")

# Sort features by absolute effect magnitude
effects_df = effects_df.sort_values(by='avg_effect', key=abs, ascending=False)

# Plot average effect for each feature
plt.figure(figsize=(8, 6))
colors = effects_df['avg_effect'].apply(lambda x: 'red' if x > 0 else 'blue')
plt.barh(effects_df['feature'], effects_df['avg_effect'], color=colors)
plt.xlabel("Average Effect on Predicted Probability for Days to Clearance")
plt.title("Top 20 Features Average Effect (LightGBM) - Visit 2")
plt.axvline(0, color='black', linewidth=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Decision Tree 

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.metrics import accuracy_score
import numpy as np

# Split the data into training and testing sets
X = df.drop(cols_X, axis=1)  # make sure cols_X is defined
y = df['buckets']

# Split into train+val and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the parameter distribution for hyperparameter tuning
param_dist = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': [None, 'sqrt', 'log2'],  # removed 'auto'
    'splitter': ['best', 'random']
}

n_iter = 50
param_list = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=42))

best_score = 0
best_params = None

# Manual random search loop
for params in param_list:
    clf = DecisionTreeClassifier(random_state=42, **params)
    clf.fit(X_train, y_train)
    y_preds = clf.predict(X_test)
    
    acc = accuracy_score(y_test, y_preds)
    
    if acc > best_score:
        best_score = acc
        best_params = params

print("\nBest hyperparameters found:", best_params)
print("Best validation accuracy:", best_score)


In [ ]:
# --- Sandbox leave-one-out using tuned params (removal of treatment present) ---
clf_best = DecisionTreeClassifier(random_state=42, **best_params)

correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]
    
    clf_best.fit(X_train, y_train)
    pred = clf_best.predict(X_test)[0]
    
    if pred == y_test:
        correct += 1

accuracy = correct / total
print("Leave-One-Out Accuracy:", accuracy)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import pandas as pd

# --- Prepare for Leave-One-Out Cross-Validation ---
y_true_all = []
y_pred_all = []

X = df.drop(cols_X, axis=1)
y = df['buckets']
total = len(df)

for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]

    model = DecisionTreeClassifier(random_state=42, **best_params)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)[0]

    y_true_all.append(y_test)
    y_pred_all.append(pred)

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

# === POINT ESTIMATES ===
accuracy = accuracy_score(y_true_all, y_pred_all)
precision = precision_score(y_true_all, y_pred_all, zero_division=0)
recall = recall_score(y_true_all, y_pred_all, zero_division=0)
f1 = f1_score(y_true_all, y_pred_all, zero_division=0)

tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
sensitivity = recall  # same value

# === BOOTSTRAP STANDARD DEVIATIONS ===
rng = np.random.default_rng(42)
B = 1000

acc_b, prec_b, rec_b, f1_b, spec_b = [], [], [], [], []

for _ in range(B):
    idx = rng.integers(0, len(y_true_all), len(y_true_all))
    yt = y_true_all[idx]
    yp = y_pred_all[idx]

    acc_b.append(accuracy_score(yt, yp))
    prec_b.append(precision_score(yt, yp, zero_division=0))
    rec_b.append(recall_score(yt, yp, zero_division=0))
    f1_b.append(f1_score(yt, yp, zero_division=0))

    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    spec_b.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

# Standard deviations
accuracy_std = np.std(acc_b)
precision_std = np.std(prec_b)
recall_std = np.std(rec_b)
f1_std = np.std(f1_b)
specificity_std = np.std(spec_b)

# === OUTPUT ===
print("\nDecision Tree Leave-One-Out Results (Mean ± SD):")
print(f"Accuracy: {accuracy:.4f} ± {accuracy_std:.4f}")
print(f"Precision: {precision:.4f} ± {precision_std:.4f}")
print(f"Recall: {recall:.4f} ± {recall_std:.4f}")
print(f"F1 Score: {f1:.4f} ± {f1_std:.4f}")
print(f"Specificity: {specificity:.4f} ± {specificity_std:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# Recreate model with best parameters found earlier (removal of treatment present)
clf_best = DecisionTreeClassifier(random_state=42, **best_params)

correct = 0
total = len(df)
X = df.drop(cols_X, axis=1)
y = df['buckets']

# initialize accumulator for feature importance
feature_importance_sum = np.zeros(X.shape[1])

for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    X_test = X.iloc[[i]]       # keep as DataFrame
    y_test = y.iloc[[i]]       # keep as Series
    
    # Re-initialize model in each loop
    model = DecisionTreeClassifier(random_state=42, **best_params)
    model.fit(X_train, y_train)
    
    # accumulate feature importance
    feature_importance_sum += model.feature_importances_
    
    # prediction for accuracy
    pred = model.predict(X_test)[0]
    if pred == y_test.values[0]:
        correct += 1

    # progress tracker
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"Progress: {i+1}/{total} ({(i+1)/total:.1%}) complete")

accuracy = correct / total

# average feature importance
feature_importance_avg = feature_importance_sum / total

# create DataFrame for inspection
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance_avg
}).sort_values(by="Importance", ascending=False)

print("\nDecision Tree Leave-One-Out Accuracy:", accuracy)
# print(feature_importance_df)




In [ ]:
top_20_features.head(20)

In [ ]:
# import sys
# !{sys.executable} -m pip install --upgrade scikit-learn

In [ ]:
from sklearn.tree import DecisionTreeClassifier
# Import metrics required for comprehensive evaluation (removal of treatment present)
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
import numpy as np
import pandas as pd # Already imported earlier, but good practice to keep relevant imports in function scope

# --- Configuration assumed from earlier cells ---
# clf_best is assumed to be defined in an earlier cell, but we need its parameters.
# Assuming best_params is defined from the hyperparameter tuning step.
# Assuming df and cols_X are defined.

# --- Prepare for Leave-One-Out Cross-Validation ---
# To store all true and predicted labels for comprehensive metrics calculation
y_true_all = []
y_pred_all = []
correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']
# ----------------------------------------------------

for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]

    # Re-initialize model in each loop to ensure an independent test in LOOCV
    # The actual best model found from tuning should be used here.
    model = DecisionTreeClassifier(random_state=42, **best_params)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)[0]

    # Collect true and predicted values
    y_true_all.append(y_test)
    y_pred_all.append(pred)

    if pred == y_test:
        correct += 1
        
# Calculate final accuracy
accuracy = correct / total

# === COMPREHENSIVE METRICS CALCULATION (Positive Class: 1) ===

# Precision, Recall (Sensitivity), and F1 Score
# 'binary' average assumes y=1 is the positive class.
precision = precision_score(y_true_all, y_pred_all, average='binary', zero_division=0)
recall = recall = recall_score(y_true_all, y_pred_all, average='binary', zero_division=0) # Recall is the same as Sensitivity

# F1 Score
f1 = f1_score(y_true_all, y_pred_all, average='binary', zero_division=0)

# Calculate Specificity (True Negative Rate) and Sensitivity (True Positive Rate/Recall)
# Assumes binary classification with labels 0 (negative) and 1 (positive)
tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()

# Specificity (True Negative Rate) = TN / (TN + FP)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Sensitivity (True Positive Rate) = TP / (TP + FN)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0


print("\nDecision Tree Leave-One-Out Results:")
print(f"Leave-One-Out Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# The requested calculation code for specificity and sensitivity:
print("\n# === Specificity Calculation ===")
# Assumes binary classification with labels 0 (negative) and 1 (positive)
# Note: tn, fp, fn, tp are already defined above.
# We skip re-defining 'specificity' here to avoid variable shadowing and just print the existing one.
print("Specificity (True Negative Rate):", specificity)

# from sklearn.metrics import confusion_matrix # Already imported above

# Assumes binary classification with labels 0 (negative) and 1 (positive)
# Note: tn, fp, fn, tp are already defined above.
# We skip re-defining 'sensitivity' here to avoid variable shadowing and just print the existing one.
print("Sensitivity (True Positive Rate):", sensitivity)

# The original LOOCV logic does not involve feature importance dataframe creation, 
# so we remove the unused feature importance accumulator and calculation to keep the code clean.
# I will keep the original file's Decision Tree code in snippet 7 with the necessary updates.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#(removal of treatment present)
# Compute confusion matrix
cm = confusion_matrix(y_true_all, y_pred_all)
tn, fp, fn, tp = cm.ravel()

# Convert to a labeled DataFrame for better readability
cm_df = pd.DataFrame(cm, 
                     index=[' 0', ' 1'], 
                     columns=[' 0', ' 1'])

plt.figure(figsize=(5,4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Decision Tree')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

# Prepare data
X = df.drop(cols_X, axis=1)
y = df['buckets']

total = len(df)
correct = 0

# Initialize accumulator for feature importance
feature_importance_sum = np.zeros(X.shape[1])

# Leave-one-out loop
for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[[i]]
    
    # Train Decision Tree
    model = DecisionTreeClassifier(random_state=42, **best_params)
    model.fit(X_train, y_train)
    
    # Accumulate feature importance
    feature_importance_sum += model.feature_importances_
    
    # Prediction for accuracy
    pred = model.predict(X_test)[0]
    if pred == y_test.values[0]:
        correct += 1
    
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"Progress: {i+1}/{total} ({(i+1)/total:.1%}) complete")

# Compute averages
accuracy = correct / total
feature_importance_avg = feature_importance_sum / total

# Create DataFrame for feature importance
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance_avg
}).sort_values(by='importance', ascending=False)

top_20_features = feature_importance_df.head(20)
top_features = top_20_features['feature'].tolist()

# Compute average effect per top feature, replacing empty subset with 0
effects = []
for f in top_features:
    median = X[f].median()
    
    # Split into high/low subsets
    X_high = X[X[f] > median]
    X_low  = X[X[f] <= median]
    
    # Compute probabilities; replace empty group with 0
    prob_high = model.predict_proba(X_high)[:, 1].mean() if not X_high.empty else 0.0
    prob_low  = model.predict_proba(X_low)[:, 1].mean() if not X_low.empty else 0.0
    
    # Compute average effect
    avg_effect = prob_high - prob_low
    effects.append({
        'feature': f,
        'avg_effect': avg_effect,
        'direction': 'toward 1' if avg_effect > 0 else 'toward 0'
    })

effects_df = pd.DataFrame(effects)

# Handle case where no valid features are found
if effects_df.empty:
    raise ValueError("No valid features found for effect calculation. Check your data.")

# Sort and plot
effects_df = effects_df.sort_values(by='avg_effect', key=abs, ascending=False)
plt.figure(figsize=(8,6))
colors = effects_df['avg_effect'].apply(lambda x: 'red' if x > 0 else 'blue')
plt.barh(effects_df['feature'], effects_df['avg_effect'], color=colors)
plt.xlabel("Average Effect on Predicted Probability for Days to Clearance")
plt.title("Top 20 Features Average Effect (Decision Tree) - Visit 2")
plt.axvline(0, color='black', linewidth=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Random Forrest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.metrics import accuracy_score

# Split the data into training and testing sets
X = df.drop(cols_X, axis=1)  # make sure cols_X is defined (dropp treatment present)
y = df['buckets']

# Split into train+val and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_dist = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'max_features': [None, 'sqrt', 'log2']  # removed 'auto'
}

n_iter = 50
param_list = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=42))

best_score = 0
best_params = None

# Manual random search loop
for params in param_list:
    rf_clf = RandomForestClassifier(random_state=42, **params)
    rf_clf.fit(X_train, y_train)
    y_preds = rf_clf.predict(X_test)  # use rf_clf here
    
    acc = accuracy_score(y_test, y_preds)
    
    if acc > best_score:
        best_score = acc
        best_params = params

print("\nBest hyperparameters found:", best_params)
print("Best validation accuracy:", best_score)


In [ ]:
# --- Sandbox leave-one-out using tuned params (dropp treatment present)---
rf_clf_best = RandomForestClassifier(random_state=42, **best_params)

correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]
    
    rf_clf_best.fit(X_train, y_train)
    pred = rf_clf_best.predict(X_test)[0]
    
    if pred == y_test:
        correct += 1

accuracy = correct / total
print("Leave-One-Out Accuracy:", accuracy)

In [ ]:

correct = 0
total = len(df)
X = df.drop(cols_X, axis=1)
y = df['buckets']

# initialize accumulator for feature importance
feature_importance_sum = np.zeros(X.shape[1])

for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    X_test = X.iloc[[i]]       # keep as DataFrame
    y_test = y.iloc[[i]]       # keep as Series
    
    # Re-initialize model in each loop
    rf_clf_best = RandomForestClassifier(random_state=42, **best_params)
    rf_clf_best.fit(X_train, y_train)
    
    # accumulate feature importance
    feature_importance_sum += rf_clf_best.feature_importances_
    
    # prediction for accuracy
    pred = rf_clf_best.predict(X_test)[0]
    if pred == y_test.values[0]:
        correct += 1

    # progress tracker
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"Progress: {i+1}/{total} ({(i+1)/total:.1%}) complete")

accuracy = correct / total

# average feature importance
feature_importance_avg = feature_importance_sum / total

# create DataFrame for inspection
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance_avg
}).sort_values(by="Importance", ascending=False)

print("\Random Forrest Leave-One-Out Accuracy:", accuracy)
# print(feature_importance_df)


In [ ]:
# metrics recall, f1, precision, specificity, sensitivity (dropp treatment present)

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
# Import metrics required for comprehensive evaluation
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# --- Prepare for Leave-One-Out Cross-Validation ---
# To store all true and predicted labels for comprehensive metrics calculation
y_true_all = [] # New variable to store all true labels
y_pred_all = [] # New variable to store all predicted labels
correct = 0
total = len(df)
X = df.drop(cols_X, axis=1)
y = df['buckets']

# initialize accumulator for feature importance (kept from original code)
feature_importance_sum = np.zeros(X.shape[1])
# ----------------------------------------------------

for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    X_test = X.iloc[[i]]       # keep as DataFrame
    y_test = y.iloc[[i]]       # keep as Series

    # Re-initialize model in each loop
    rf_clf_best = RandomForestClassifier(random_state=42, **best_params)
    rf_clf_best.fit(X_train, y_train)

    # accumulate feature importance (kept from original code)
    feature_importance_sum += rf_clf_best.feature_importances_

    # prediction for accuracy
    pred = rf_clf_best.predict(X_test)[0]
    
    # === Store True and Predicted Labels for Metrics ===
    y_true_all.append(y_test.values[0])
    y_pred_all.append(pred)
    # ===================================================

    if pred == y_test.values[0]:
        correct += 1

    # progress tracker (kept from original code)
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"Progress: {i+1}/{total} ({(i+1)/total:.1%}) complete")

# accuracy = correct / total



# === COMPREHENSIVE METRICS CALCULATION (Positive Class: 1) ===

# Precision, Recall (Sensitivity), and F1 Score
# Assumes binary classification and '1' is the positive class.
precision = precision_score(y_true_all, y_pred_all, average='binary', zero_division=0)
recall = recall_score(y_true_all, y_pred_all, average='binary', zero_division=0) # Recall is the same as Sensitivity
f1 = f1_score(y_true_all, y_pred_all, average='binary', zero_division=0)

# Calculate Specificity (True Negative Rate) and Sensitivity (True Positive Rate/Recall)
# Get the components of the confusion matrix: True Negative, False Positive, False Negative, True Positive
tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()

# Specificity (True Negative Rate) = TN / (TN + FP)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Sensitivity (True Positive Rate/Recall) = TP / (TP + FN)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print("\nRandom Forest Leave-One-Out Results:")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Specificity: {specificity:.4f}")



In [ ]:
# Compute confusion matrix (dropp treatment present)
cm = confusion_matrix(y_true_all, y_pred_all)
tn, fp, fn, tp = cm.ravel()

# Convert to a labeled DataFrame for better readability
cm_df = pd.DataFrame(cm, 
                     index=[' 0', ' 1'], 
                     columns=[' 0', ' 1'])

plt.figure(figsize=(5,4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Forrest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


In [ ]:
print(f"Sensitivity: {sensitivity:.4f}")

In [ ]:
feature_importance_df.head(20) #(dropp treatment present)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier  # Random Forest

# Prepare data
X = df.drop(cols_X, axis=1)
y = df['buckets']

total = len(df)
correct = 0

# Initialize accumulator for feature importance
feature_importance_sum = np.zeros(X.shape[1])

# Leave-one-out loop
for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[[i]]
    
    # Train Random Forest
    model = RandomForestClassifier(random_state=42, **best_params)
    model.fit(X_train, y_train)
    
    # Accumulate feature importance
    feature_importance_sum += model.feature_importances_
    
    # Prediction for accuracy
    pred = model.predict(X_test)[0]
    if pred == y_test.values[0]:
        correct += 1
    
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"Progress: {i+1}/{total} ({(i+1)/total:.1%}) complete")

# Compute averages
accuracy = correct / total
feature_importance_avg = feature_importance_sum / total

# Create DataFrame for feature importance
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance_avg
}).sort_values(by='importance', ascending=False)

top_20_features = feature_importance_df.head(20)
top_features = top_20_features['feature'].tolist()

# Compute average effect per top feature, replacing empty subset with 0
effects = []
for f in top_features:
    median = X[f].median()
    
    X_high = X[X[f] > median]
    X_low  = X[X[f] <= median]
    
    # Compute probabilities, replace empty group with 0
    prob_high = model.predict_proba(X_high)[:, 1].mean() if not X_high.empty else 0.0
    prob_low  = model.predict_proba(X_low)[:, 1].mean() if not X_low.empty else 0.0
    
    avg_effect = prob_high - prob_low
    
    effects.append({
        'feature': f,
        'avg_effect': avg_effect,
        'direction': 'toward 1' if avg_effect > 0 else 'toward 0'
    })

effects_df = pd.DataFrame(effects)
effects_df = effects_df.sort_values(by='avg_effect', key=abs, ascending=False)

# Plot average effect ("bee plot")
plt.figure(figsize=(8,6))
colors = effects_df['avg_effect'].apply(lambda x: 'red' if x > 0 else 'blue')
plt.barh(effects_df['feature'], effects_df['avg_effect'], color=colors)
plt.xlabel("Average Effect on Predicted Probability for Days to Clearance")
plt.title("Top 20 Features Average Effect (Random Forest) - Visit 2")
plt.axvline(0, color='black', linewidth=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## XGBoost

In [ ]:
pip install xgboost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, KFold, cross_val_predict, train_test_split
from scipy import stats
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.metrics import accuracy_score

# Split the data
X = df.drop(cols_X, axis=1)
y = df['buckets']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pure Python parameter distributions (no numpy types)
param_dist = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.85, 1.0],
    'n_estimators': [50, 100, 150],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'gamma': [0, 0.1, 0.3],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0.5, 1.0, 1.5]
}

n_iter = 50
param_list = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=42))

best_score = 0
best_params = None

# Manual random search loop
for params in param_list:
    xgb = XGBClassifier(random_state=42, **params)
    xgb.fit(X_train, y_train)
    y_preds = xgb.predict(X_test)  # use rf_clf here
    
    acc = accuracy_score(y_test, y_preds)
    
    if acc > best_score:
        best_score = acc
        best_params = params

print("\nBest hyperparameters found:", best_params)
print("Best validation accuracy:", best_score)


In [ ]:
#sandbox
xgb_best = XGBClassifier(random_state=42, **best_params)

correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]
    
    xgb_best.fit(X_train, y_train)
    pred = xgb_best.predict(X_test)[0]
    
    if pred == y_test:
        correct += 1

accuracy = correct / total
accuracy

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

xgb_best = XGBClassifier(random_state=42, **best_params)

correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

# container for feature importance
importance_accum = pd.DataFrame(0, index=X.columns, columns=["importance"])

for i in range(total):
    # Split
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]
    
    # Fit
    xgb_best.fit(X_train, y_train)
    pred = xgb_best.predict(X_test)[0]
    
    # Track accuracy
    if pred == y_test:
        correct += 1
    
    # Collect feature importance (gain-based)
    fold_importance = xgb_best.get_booster().get_score(importance_type="gain")
    for feat, score in fold_importance.items():
        importance_accum.loc[feat, "importance"] += score

# Final results
accuracy = correct / total
importance_accum["importance"] /= total   # average across folds
importance_accum = importance_accum.sort_values(by="importance", ascending=False)

# print("LOO Accuracy:", accuracy)
print("\nAverage Feature Importance (Gain):")
print(importance_accum)


In [ ]:
# The following code implements Leave-One-Out Cross-Validation (LOOCV)
# for an XGBoostClassifier. It measures model performance using:
# - Accuracy (Overall correct predictions)
# - Precision (Model's positive predictions that were actually correct)
# - Recall / Sensitivity (Actual positive cases correctly identified)
# - F1 Score (Harmonic mean of Precision and Recall)
# - Specificity (Actual negative cases correctly identified)
# Additionally, it calculates the average feature importance across all folds.

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
# Import metrics required for comprehensive evaluation
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

xgb_best = XGBClassifier(random_state=42, **best_params)

# === New variables to store true and predicted labels ===
y_true_all = []
y_pred_all = []
# =======================================================
correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

# container for feature importance
importance_accum = pd.DataFrame(0, index=X.columns, columns=["importance"])

for i in range(total):
    # Split
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i] # y_test is a single value Series
    
    # Fit
    xgb_best.fit(X_train, y_train)
    pred = xgb_best.predict(X_test)[0] # pred is a single value

    # === Collect True and Predicted Labels for Metrics ===
    y_true_all.append(y_test)
    y_pred_all.append(pred)
    # =======================================================
    
    # Track accuracy
    if pred == y_test:
        correct += 1
    
    # Collect feature importance (gain-based)
    fold_importance = xgb_best.get_booster().get_score(importance_type="gain")
    for feat, score in fold_importance.items():
        importance_accum.loc[feat, "importance"] += score

# Final results
accuracy = correct / total
importance_accum["importance"] /= total    # average across folds
importance_accum = importance_accum.sort_values(by="importance", ascending=False)


# === COMPREHENSIVE METRICS CALCULATION (Positive Class: 1) ===

# Precision, Recall (Sensitivity), and F1 Score
# Assumes binary classification and '1' is the positive class.
precision = precision_score(y_true_all, y_pred_all, average='binary', zero_division=0)
recall = recall_score(y_true_all, y_pred_all, average='binary', zero_division=0) # Recall is the same as Sensitivity
f1 = f1_score(y_true_all, y_pred_all, average='binary', zero_division=0)

# Calculate Specificity (True Negative Rate) and Sensitivity (True Positive Rate/Recall)
# Get the components of the confusion matrix: True Negative, False Positive, False Negative, True Positive
tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()

# Specificity (True Negative Rate) = TN / (TN + FP)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Sensitivity (True Positive Rate/Recall) = TP / (TP + FN)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print("\nXGBoost Leave-One-Out Results:")
print(f"Leave-One-Out Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Specificity: {specificity:.4f}")
# print("\nAverage Feature Importance (Gain):")
# print(importance_accum)

In [ ]:
# Compute confusion matrix (dropp treatment present)
cm = confusion_matrix(y_true_all, y_pred_all)
tn, fp, fn, tp = cm.ravel()

# Convert to a labeled DataFrame for better readability
cm_df = pd.DataFrame(cm, 
                     index=[' 0', ' 1'], 
                     columns=[' 0', ' 1'])

plt.figure(figsize=(5,4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - XGBoost')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


In [ ]:
importance_accum.head(20)

In [ ]:

top_20_features

In [ ]:
top_20_features = importance_accum.head(20)

top_20_features = (
    top_20_features
    .reset_index()
    .rename(columns={'index': 'features'})
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

# # Prepare data
# X = df.drop(cols_X, axis=1)
# y = df['buckets']

# total = len(df)
# correct = 0

# # Initialize accumulator for feature importance
# feature_importance_sum = np.zeros(X.shape[1])

# # Leave-One-Out cross-validation
# for i in range(total):
#     X_train = X.drop(index=i)
#     y_train = y.drop(index=i)
#     X_test = X.iloc[[i]]
#     y_test = y.iloc[[i]]
    
#     # Train XGBoost model
#     model = XGBClassifier(random_state=42, **best_params)
#     model.fit(X_train, y_train)
    
#     # Accumulate feature importance (gain-based)
#     booster = model.get_booster()
#     importance_dict = booster.get_score(importance_type='gain')
#     importance_values = np.array([importance_dict.get(f"f{j}", 0) for j in range(X.shape[1])])
#     feature_importance_sum += importance_values
    
#     # Prediction for accuracy
#     pred = model.predict(X_test)[0]
#     if pred == y_test.values[0]:
#         correct += 1
    
#     if (i + 1) % 50 == 0 or (i + 1) == total:
#         print(f"Progress: {i+1}/{total} ({(i+1)/total:.1%}) complete")

# # Compute averages
# accuracy = correct / total
# feature_importance_avg = feature_importance_sum / total

# # Create DataFrame for feature importance
# feature_importance_df = pd.DataFrame({
#     'feature': X.columns,
#     'importance': feature_importance_avg
# }).sort_values(by='importance', ascending=False)

# Select top 20 features



top_features = top_20_features['features'].tolist()

# Compute average effect per top feature, handling empty groups
effects = []
for f in top_features:
    median = X[f].median()
    
    # Split into high and low
    X_high = X[X[f] > median]
    X_low  = X[X[f] <= median]
    
    # If either group is empty, replace its predicted probability with 0
    prob_high = model.predict_proba(X_high)[:, 1].mean() if not X_high.empty else 0.0
    prob_low  = model.predict_proba(X_low)[:, 1].mean() if not X_low.empty else 0.0
    
    avg_effect = prob_high - prob_low
    effects.append({
        'feature': f,
        'avg_effect': avg_effect,
        'direction': 'toward 1' if avg_effect > 0 else 'toward 0'
    })

effects_df = pd.DataFrame(effects)
effects_df = effects_df.sort_values(by='avg_effect', key=abs, ascending=False)

# Plot bee plot
plt.figure(figsize=(8,6))
colors = effects_df['avg_effect'].apply(lambda x: 'red' if x > 0 else 'blue')
plt.barh(effects_df['feature'], effects_df['avg_effect'], color=colors)
plt.xlabel("Average Effect on Predicted Probability for Days to Clearance")
plt.title("Top 20 Features Average Effect (XGBoost) - Visit 2")
plt.axvline(0, color='black', linewidth=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## SVC

In [ ]:
from sklearn.svm import SVC 
from sklearn.model_selection import cross_val_score, KFold, cross_val_predict, train_test_split
#hyperparameter tuning (dropp treatment present)

# Split the data into training and testing sets
X = df.drop(cols_X, axis=1)
y = df['buckets']

# Split into train+val and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score


from sklearn.model_selection import train_test_split, RandomizedSearchCV, ParameterSampler

# Create the model 
model = SVC()

param_dist_svc = {
    'C': [0.1, 10, 100],
    'gamma': [1, 0.01,0.0001],
    'kernel':['linear','rbf', 'poly']
}

n_iter = 50
param_list = list(ParameterSampler(param_dist_svc, n_iter=n_iter, random_state=42))

best_score = 0
best_params = None

# Manual random search loop
for params in param_list:
    svc_best = SVC(random_state=42, **params)
    svc_best.fit(X_train, y_train)
    y_preds = svc_best.predict(X_test)  
    
    acc = accuracy_score(y_test, y_preds)
    
    if acc > best_score:
        best_score = acc
        best_params = params

print("\nBest hyperparameters found:", best_params)
print("Best accuracy:", best_score)

In [ ]:
#sandbox (dropp treatment present)
svc_best = SVC(random_state=42, **best_params)

correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]
    
    svc_best.fit(X_train, y_train)
    pred = svc_best.predict(X_test)[0]
    
    if pred == y_test:
        correct += 1

accuracy = correct / total
accuracy

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC

svc_best = SVC(random_state=42, **best_params)

correct = 0
total = len(df)

X = df.drop(cols_X, axis=1).values
y = df['buckets'].values

coef_accum = np.zeros(X.shape[1])

for i in range(total):
    # Split
    X_train = np.delete(X, i, axis=0)
    y_train = np.delete(y, i)
    X_test = X[[i]]
    y_test = y[i]
    
    # Fit
    svc_best.fit(X_train, y_train)
    pred = svc_best.predict(X_test)[0]
    
    # Accuracy
    if pred == y_test:
        correct += 1
    
    # Feature importance = coefficients
    coef_accum += svc_best.coef_[0]

# Final results
accuracy = correct / total
avg_importance = coef_accum / total

importance_df = pd.DataFrame({
    "feature": df.drop(cols_X, axis=1).columns,
    "importance": np.abs(avg_importance)   # absolute value = strength
}).sort_values(by="importance", ascending=False)

# print("LOO Accuracy:", accuracy)
# print("\nAverage Feature Importance (Linear SVC Coefficients):")
# print(importance_df)


In [ ]:
# The following code implements Leave-One-Out Cross-Validation (LOOCV) (dropp treatment present)
# for a Support Vector Classifier (SVC). It measures model performance using:
# - Accuracy (Overall correct predictions)
# - Precision (Model's positive predictions that were actually correct)
# - Recall / Sensitivity (Actual positive cases correctly identified)
# - F1 Score (Harmonic mean of Precision and Recall)
# - Specificity (Actual negative cases correctly identified)

import numpy as np
import pandas as pd
from sklearn.svm import SVC # Assuming this import is needed based on SVC usage
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score # Import metrics

# --- Initialization ---
svc_best = SVC(random_state=42, **best_params)

# Variables to store true and predicted labels for metrics calculation
y_true_all = []
y_pred_all = []
correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

# --- LOOCV Loop ---
for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]
    
    # Fit model and predict
    svc_best.fit(X_train, y_train)
    pred = svc_best.predict(X_test)[0]
    
    # Accumulate true and predicted labels
    y_true_all.append(y_test)
    y_pred_all.append(pred)
    
    # Track accuracy
    if pred == y_test:
        correct += 1

# --- Final Results Calculation ---
accuracy = correct / total

# === COMPREHENSIVE METRICS CALCULATION (Assuming Positive Class: 1) ===

# Precision, Recall (Sensitivity), and F1 Score
# 'binary' average assumes '1' is the positive class.
precision = precision_score(y_true_all, y_pred_all, average='binary', zero_division=0)
recall = recall_score(y_true_all, y_pred_all, average='binary', zero_division=0) # Recall is Sensitivity
f1 = f1_score(y_true_all, y_pred_all, average='binary', zero_division=0)

# Specificity (True Negative Rate) and Sensitivity (True Positive Rate/Recall)
# Get the components of the confusion matrix: True Negative (tn), False Positive (fp), False Negative (fn), True Positive (tp)
tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()

# Specificity = TN / (TN + FP)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Sensitivity = TP / (TP + FN) (same as Recall)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

# --- Print Results ---
print("\nSVC Leave-One-Out Results:")
print(f"Leave-One-Out Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Specificity: {specificity:.4f}")

In [ ]:
# Compute confusion matrix (dropp treatment present)
cm = confusion_matrix(y_true_all, y_pred_all)
tn, fp, fn, tp = cm.ravel()

# Convert to a labeled DataFrame for better readability
cm_df = pd.DataFrame(cm, 
                     index=[' 0', ' 1'], 
                     columns=[' 0', ' 1'])

plt.figure(figsize=(5,4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - SVC')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


In [ ]:
#(dropp treatment present)
importance_df.head(20)

In [ ]:
#bee plot

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler

# Prepare data
X = df.drop(cols_X, axis=1)
y = df['buckets']

total = len(df)
correct = 0

# Initialize accumulator for feature importance
feature_importance_sum = np.zeros(X.shape[1])

# Leave-One-Out cross-validation
for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[[i]]
    
    # Train SVC with probability=True
    model = SVC(random_state=42, probability=True, **best_params)
    model.fit(X_train, y_train)
    
    # Accumulate absolute coefficient values as feature importance
    feature_importance_sum += np.abs(model.coef_).flatten()
    
    # Predict and track accuracy
    pred = model.predict(X_test)[0]
    if pred == y_test.values[0]:
        correct += 1
    
    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"Progress: {i+1}/{total} ({(i+1)/total:.1%}) complete")

# Compute averages
accuracy = correct / total
feature_importance_avg = feature_importance_sum / total

# Create DataFrame for feature importance
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance_avg
}).sort_values(by='importance', ascending=False)

# Select top 20 features
top_20_features = feature_importance_df.head(20)
top_features = top_20_features['feature'].tolist()

# Compute average effect per top feature, handling empty groups
effects = []
for f in top_features:
    median = X[f].median()
    
    X_high = X[X[f] > median]
    X_low  = X[X[f] <= median]
    
    # Compute probabilities or scaled decision function; replace empty with 0
    if not X_high.empty and not X_low.empty:
        try:
            prob_high = model.predict_proba(X_high)[:, 1].mean()
            prob_low  = model.predict_proba(X_low)[:, 1].mean()
        except AttributeError:
            scaler = MinMaxScaler()
            scores_high = scaler.fit_transform(model.decision_function(X_high).reshape(-1,1)).mean()
            scores_low  = scaler.fit_transform(model.decision_function(X_low).reshape(-1,1)).mean()
            prob_high, prob_low = scores_high, scores_low
    else:
        prob_high = 0.0 if X_high.empty else model.predict_proba(X_high)[:, 1].mean()
        prob_low  = 0.0 if X_low.empty  else model.predict_proba(X_low)[:, 1].mean()
    
    avg_effect = prob_high - prob_low
    effects.append({
        'feature': f,
        'avg_effect': avg_effect,
        'direction': 'toward 1' if avg_effect > 0 else 'toward 0'
    })

effects_df = pd.DataFrame(effects)
effects_df = effects_df.sort_values(by='avg_effect', key=abs, ascending=False)

# Plot
plt.figure(figsize=(8,6))
colors = effects_df['avg_effect'].apply(lambda x: 'red' if x > 0 else 'blue')
plt.barh(effects_df['feature'], effects_df['avg_effect'], color=colors)
plt.xlabel("Average Effect on Predicted Probability for Days to Clearance")
plt.title("Top 20 Features Average Effect (SVC - Linear Kernel) - Visit 2")
plt.axvline(0, color='black', linewidth=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# After computing effects_df

# Print average effect for each feature
print("Average Effect Values for Top Features (SVC - Linear Kernel):\n")
for idx, row in effects_df.iterrows():
    print(f"{row['feature']}: {row['avg_effect']:.4f} ({row['direction']})")


## Lasso Regression

In [ ]:
#determine best alpha value for lasso

from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold, cross_val_score, train_test_split
import numpy as np

def find_best_alpha_with_kfold(X, y, alphas, k=5):
  """
  Finds the best alpha value for Lasso regression using k-fold cross-validation.

  Args:
    X: Feature matrix (numpy array or pandas DataFrame).
    y: Target variable (numpy array or pandas Series).
    alphas: List of alpha values to try.
    k: Number of folds for k-fold cross-validation.

  Returns:
    best_alpha: The alpha value that yields the best cross-validation score.
  """

  best_alpha = None
  best_score = -np.inf  # Initialize with negative infinity

  for alpha in alphas:
    lasso = Lasso(alpha=alpha)
    scores = cross_val_score(lasso, X, y, cv=k, scoring='neg_mean_squared_error')
    avg_score = np.mean(scores) 

    if avg_score > best_score:
      best_score = avg_score
      best_alpha = alpha

  return best_alpha

# Example Usage:
# Split the data into training and testing sets
X = df.drop(cols_X, axis=1)
y = df['buckets']
alphas = [0.001, 0.01, 0.1, 1, 10]  # Example alpha values

best_alpha = find_best_alpha_with_kfold(X, y, alphas)
print(f"Best alpha: {best_alpha}")

In [ ]:
#sandbox

correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]

    lasso = Lasso(alpha=best_alpha)
    lasso.fit(X_train, y_train)
    pred = lasso.predict(X_test)[0]
    
    if pred == y_test:
        correct += 1

accuracy = correct / total
accuracy

## Ridge Regression

Did random search method

drop treatment present

In [ ]:
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

def find_best_ridge_random_subset(X, y, alphas, n_iter=50, k=10, random_state=None):
    """
    Randomly samples alpha values from a predefined list and finds the best estimator 
    using k-fold cross-validation.

    Args:
        X: Feature matrix (numpy array or pandas DataFrame).
        y: Target variable (numpy array or pandas Series).
        alphas: List of candidate alpha values.
        n_iter: Number of alpha values to sample and test.
        k: Number of folds for k-fold cross-validation.
        random_state: Optional random seed for reproducibility.

    Returns:
        best_alpha: The alpha value with the best average cross-validation score.
        best_estimator: The fitted RidgeClassifier with best_alpha.
        best_score: The best CV score achieved.
    """
    rng = np.random.default_rng(random_state)
    sampled_alphas = rng.choice(alphas, size=min(n_iter, len(alphas)), replace=False)

    best_alpha = None
    best_score = -np.inf
    best_estimator = None

    for alpha in sampled_alphas:
        ridge = RidgeClassifier(alpha=alpha)
        scores = cross_val_score(ridge, X, y, cv=k, scoring='accuracy')
        avg_score = np.mean(scores)

        if avg_score > best_score:
            best_score = avg_score
            best_alpha = alpha
            best_estimator = ridge.fit(X, y)  # refit on all data

    return best_alpha, best_estimator, best_score


# Example usage
X = df.drop(cols_X, axis=1)
y = df['buckets']
alphas = [0.001, 0.01, 0.1, 1, 10,20]

best_alpha, best_model, best_score = find_best_ridge_random_subset(
    X, y, alphas, n_iter=50, k=10, random_state=42
)

print(f"Best alpha: {best_alpha}")



In [ ]:
#sandbox

correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]

    ridge = RidgeClassifier(alpha=best_alpha)
    ridge.fit(X_train, y_train)
    pred = ridge.predict(X_test)[0]
    
    if pred == y_test:
        correct += 1

accuracy = correct / total
accuracy

In [ ]:
# The following code implements Leave-One-Out Cross-Validation (LOOCV)
# for a Ridge Classifier. It measures model performance using:
# - Accuracy (Overall correct predictions)
# - Precision (Model's positive predictions that were actually correct)
# - Recall / Sensitivity (Actual positive cases correctly identified)
# - F1 Score (Harmonic mean of Precision and Recall)
# - Specificity (Actual negative cases correctly identified)

import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeClassifier # Import the classifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score # Import metrics

# --- Initialization ---
# Variables to store true and predicted labels for metrics calculation
y_true_all = []
y_pred_all = []
correct = 0
total = len(df)

X = df.drop(cols_X, axis=1)
y = df['buckets']

# --- LOOCV Loop ---
for i in range(total):
    X_train = X.drop(i)
    y_train = y.drop(i)
    X_test = X.iloc[[i]]
    y_test = y.iloc[i]

    # Initialize and fit the classifier with the best_alpha (assumed to be defined)
    ridge = RidgeClassifier(alpha=best_alpha)
    ridge.fit(X_train, y_train)
    pred = ridge.predict(X_test)[0]
    
    # Accumulate true and predicted labels
    y_true_all.append(y_test)
    y_pred_all.append(pred)
    
    # Track accuracy
    if pred == y_test:
        correct += 1

# --- Final Results Calculation ---
accuracy = correct / total

# === COMPREHENSIVE METRICS CALCULATION (Assuming Positive Class: 1) ===

# Precision, Recall (Sensitivity), and F1 Score
# 'binary' average assumes '1' is the positive class.
precision = precision_score(y_true_all, y_pred_all, average='binary', zero_division=0)
recall = recall_score(y_true_all, y_pred_all, average='binary', zero_division=0) # Recall is Sensitivity
f1 = f1_score(y_true_all, y_pred_all, average='binary', zero_division=0)

# Specificity (True Negative Rate) and Sensitivity (True Positive Rate/Recall)
# Get the components of the confusion matrix: TN, FP, FN, TP
# .ravel() flattens the 2x2 matrix into a 4-item tuple in TN, FP, FN, TP order.
tn, fp, fn, tp = confusion_matrix(y_true_all, y_pred_all).ravel()

# Specificity = TN / (TN + FP)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

# Sensitivity = TP / (TP + FN) (same as Recall)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

# --- Output the desired metrics ---
print("\nRidge Classifier Leave-One-Out Results:")
print(f"Leave-One-Out Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Specificity: {specificity:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Compute confusion matrix (dropp treatment present)
cm = confusion_matrix(y_true_all, y_pred_all)
tn, fp, fn, tp = cm.ravel()

# Convert to a labeled DataFrame for better readability
cm_df = pd.DataFrame(cm, 
                     index=[' 0', ' 1'], 
                     columns=[' 0', ' 1'])

plt.figure(figsize=(5,4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Ridge')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()


In [ ]:
from sklearn.linear_model import RidgeClassifier
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np

# Fit final model with best_alpha
ridge = RidgeClassifier(alpha=best_alpha)
ridge.fit(X, y)

# Compute permutation importance
perm = permutation_importance(
    ridge,
    X,
    y,
    scoring="accuracy",
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)

# Create DataFrame
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": perm.importances_mean
}).sort_values(by="importance", ascending=False)

print("\nFeature importance (Permutation Importance):")
importance_df.head(20)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import MinMaxScaler

# Prepare data
X = df.drop(cols_X, axis=1)
y = df['buckets']

total = len(df)
correct = 0

# Initialize accumulator for permutation importance
feature_importance_sum = np.zeros(X.shape[1])

# Leave-One-Out cross-validation
for i in range(total):
    X_train = X.drop(index=i)
    y_train = y.drop(index=i)
    X_test  = X.iloc[[i]]
    y_test  = y.iloc[[i]]

    # Train Ridge Classifier
    ridge = RidgeClassifier(alpha=best_alpha)
    ridge.fit(X_train, y_train)

    # ---- PERMUTATION IMPORTANCE (REPLACEMENT) ----
    perm = permutation_importance(
        ridge,
        X_train,
        y_train,
        scoring="accuracy",
        n_repeats=10,
        random_state=42,
        n_jobs=-1
    )

    feature_importance_sum += perm.importances_mean

    # Prediction for accuracy
    pred = ridge.predict(X_test)[0]
    if pred == y_test.values[0]:
        correct += 1

    if (i + 1) % 50 == 0 or (i + 1) == total:
        print(f"Progress: {i+1}/{total} ({(i+1)/total:.1%}) complete")

# Compute averages
accuracy = correct / total
feature_importance_avg = feature_importance_sum / total

# Create DataFrame for feature importance
feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': feature_importance_avg
}).sort_values(by='importance', ascending=False)

# Select top 20 features
top_20_features = feature_importance_df.head(20)
top_features = top_20_features['feature'].tolist()

# ---- REFIT FINAL MODEL FOR EFFECT DIRECTION ----
ridge = RidgeClassifier(alpha=best_alpha)
ridge.fit(X, y)

# Compute average effect per top feature safely
effects = []
for f in top_features:
    median = X[f].median()

    X_high = X[X[f] > median]
    X_low  = X[X[f] <= median]

    # Compute decision function scores and handle empty groups
    scaler = MinMaxScaler()
    if not X_high.empty:
        scores_high = scaler.fit_transform(ridge.decision_function(X_high).reshape(-1,1)).mean()
    else:
        scores_high = 0.0

    if not X_low.empty:
        scores_low = scaler.fit_transform(ridge.decision_function(X_low).reshape(-1,1)).mean()
    else:
        scores_low = 0.0

    avg_effect = scores_high - scores_low
    effects.append({
        'feature': f,
        'avg_effect': avg_effect,
        'direction': 'toward 1' if avg_effect > 0 else 'toward 0'
    })

effects_df = pd.DataFrame(effects)
effects_df = effects_df.sort_values(by='avg_effect', key=abs, ascending=False)

# Print average effect for each feature
print("Average Effect Values for Top Features (Ridge Classifier):\n")
for idx, row in effects_df.iterrows():
    print(f"{row['feature']}: {row['avg_effect']:.4f} ({row['direction']})")

# Plot
plt.figure(figsize=(8,6))
colors = effects_df['avg_effect'].apply(lambda x: 'red' if x > 0 else 'blue')
plt.barh(effects_df['feature'], effects_df['avg_effect'], color=colors)
plt.xlabel("Average Effect on Predicted Probability for Days to Clearance")
plt.title("Top 20 Features Average Effect (Ridge Regression) - Visit 2")
plt.axvline(0, color='black', linewidth=0.8)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
